一、多样化长剧本生成

In [1]:
import json
import os
import time
from openai import OpenAI

API_KEY = "sk-732017ac4c85439fb123c6a954f124cd"
client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com/v1")
DIALOGUES_FILE = "dialogues_dataset_100.json"
METADATA_FILE = "world_settings_backup_100.json"

THEMES =[
    "仙侠：蜀山剑派百年大考，掌门遇刺", "仙侠：上古魔尊突破封印，三界大乱", "仙侠：凡人修仙，底层坊市的法宝争夺", "仙侠：妖族天庭复苏，人族修士的末日", "仙侠：昆仑秘境寻宝，同门师兄弟反目",
    "修真：拍卖会上神秘残卷引发的截胡血案", "修真：大乘期老怪陨落后的夺舍重生", "仙侠：正道剑子与魔教圣女的禁忌之恋", "修真：天道崩塌，九大宗门瓜分气运", "仙侠：凡间帝王试图长生引发的人仙大战",
    
    "历史：唐朝·玄武门之变前夕的秦王府密谋", "历史：宋朝·靖康之耻，皇室南逃途中的绝望与背叛", "历史：明朝·土木堡之变，皇帝被俘后的朝堂动荡", 
    "历史：三国·赤壁之战前夜，周瑜与诸葛亮的暗中斗法", "历史：秦朝·荆轲刺秦王失败后的咸阳宫大清洗", "历史：汉朝·七国之乱爆发，景帝削藩的残酷博弈",
    "历史：清朝·九子夺嫡，康熙驾崩当夜的畅春园风暴", "历史：唐朝·马嵬坡兵变，安史之乱下的皇室抉择", "历史：明末·崇祯煤山自缢前的亡国前夜", "历史：三国·白帝城托孤，刘备与诸葛亮的最后交锋",
    "历史：汉朝·巫蛊之祸，太子刘据的绝地反击", "历史：春秋·卧薪尝胆，越王勾践与范蠡的复国密谋", "历史：清末·戊戌政变，谭嗣同夜访袁世凯", "历史：南北朝·侯景之乱，梁武帝饿死台城", "历史：民国·西安事变前夜的各方势力暗战",
    
    "赛博朋克：霓虹都市下的义体医生与黑客财阀恩怨", "赛博朋克：底层贫民窟的AI觉醒事件", "赛博朋克：记忆走私犯被大企业追杀", "赛博朋克：机械战警的意识残留与腐败警局", "赛博朋克：网络空间中的数字灵魂交易市场",
    "硬科幻：银河帝国边境舰队的叛乱密谋", "硬科幻：木星矿工大罢工与资本镇压", "星际科幻：世代飞船航行800年后的内部宗教分裂", "科幻：时间线巡逻特工发现历史被篡改", "科幻：发现外星高等文明遗迹后的多国联合科考队阴谋",
    "废土科幻：核冬天的地下避难所能源耗尽危机", "赛博朋克：大型全息游戏中玩家意识被困", "科幻：戴森球建造工程发生的高级文明入侵", "硬科幻：火星殖民地宣布独立的第一枪", "科幻：克隆人替代本体计划败露",
    
    "悬疑：暴风雪山庄，亿万富翁的遗产之谜", "悬疑：废弃精神病院里的午夜探险与连环杀手", "悬疑：深海豪华游轮上的密室毒杀案", "悬疑：无限循环的地铁列车，寻找唯一的生门", "悬疑：失去记忆的侦探在凶案现场醒来",
    "推理：民国上海滩百乐门头牌身亡之谜", "悬疑：封闭高中的晚自习灵异杀人事件", "推理：法医与高智商连环杀手的心理博弈", "悬疑：网络死亡直播间的幕后黑手", "悬疑：多重人格患者的精神世界法庭",
    "推理：警局内部排查黑警的48小时", "悬疑：偏僻山村买卖人口的黑暗地窖", "推理：完美不在场证明的跨国列车凶杀", "悬疑：神秘家族古堡的吸血鬼传说成真", "悬疑：剧本杀店内发生的真实谋杀案",
    
    "都市：华尔街顶级投行之间的百亿并购暗战", "都市：娱乐圈顶流明星的地下恋情与公关危机", "都市：豪门家族遗嘱公布后的兄弟阋墙", "都市：顶级律师事务所的内部合伙人斗争", "都市：跨国医药公司的特效药造假黑幕",
    "商战：互联网大厂高管争权与大规模裁员", "都市：新能源汽车造车新势力破产重组的最后博弈", "职场：公关公司处理食品安全危机的黑白手段", "商战：做空机构突袭百亿上市公司的48小时", "都市：真假千金身份曝光后的名媛圈大洗牌",
    "都市：黑客黑入全球银行系统的金融恐慌", "职场：时尚杂志主编之位的新老交替战", "都市：澳门地下赌场的千王之王对决", "都市：网红MCN机构的合同陷阱与主播反击", "商战：老字号国货品牌面临外资恶意收购",
    
    "武侠：绝世神兵现世，六大门派围攻光明顶", "武侠：天下第一剑客退隐客栈遭遇仇家追杀", "武侠：天下镖局押送神秘红镖的生死之路", "武侠：锦衣卫追杀忠良之后，东厂暗中设局", "武侠：武林盟主大寿之日突发中毒身亡",
    "武侠：魔教妖女与名门正派大弟子的逃亡", "武侠：塞外客栈，各路草莽争夺藏宝图", "武侠：少林寺藏经阁秘籍失窃引发的武林浩劫", "武侠：江南首富灭门惨案的孤儿复仇记", "武侠：朝廷招安绿林好汉引发的内部清洗",
    
    "西幻：勇者小队讨伐魔王后发现被帝国背叛", "魔法：皇家魔法学院禁林中的黑魔法复苏", "西幻：亡灵法师试图拯救被诅咒的公主", "西幻：精灵族与矮人族因神器产量的百年边界战争", "西幻：没落龙骑士家族孵化出最后一条黑龙",
    "魔法：平民天才与纯血贵族在魔法大考的对决", "西幻：光明教廷异端审判庭的内部腐败", "西幻：吸血鬼十三氏族的百年议会与狼人突袭", "魔法：大炼金术师触碰禁忌合成人造灵魂", "西幻：流浪佣兵团卷入帝国王储争夺战",
    
    "克苏鲁：南极科考队在冰层下挖出远古旧神遗迹", "规则怪谈：诡异动物园的夜班保安求生", "无限流：主神空间小队在丧尸副本面临团灭", "末日：全球海平面瞬间上升百米后的诺亚方舟夺权", "诡秘：维多利亚时代的迷雾伦敦连环剖尸案",
    "克苏鲁：偏远渔村献祭海神的黑暗庆典", "规则怪谈：入职一家员工不能看镜子的诡异公司", "无限流：中洲队与恶魔队在团战副本的智谋碰撞", "末日：全球变异，只有盲人不会被怪物感知的世界", "大逃杀：100个死囚被投放到荒岛的极限生存"
]


def generate_outline(theme):
    """阶段一：生成世界观大纲（仅作为上下文提示使用）"""
    print(f"正在构思世界观大纲：【{theme}】...")
    prompt = f"""你是一个顶级的剧本作家。请根据主题【{theme}】，设计一个包含完整世界观和剧情走向的长剧本大纲。
要求：
1. 剧本必须正好分为 5 幕（Scenes）。
2. 输出严格的 JSON 对象，包含以下字段：
{{
  "world_setting": "世界观详细背景（约150字）",
  "characters": "核心人物设定（约200字）",
  "scenes_outline":[
    "第一幕：...剧情...", "第二幕：...剧情...", "第三幕：...剧情...", "第四幕：...剧情...", "第五幕：...剧情..."
  ]
}}"""
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"},
            temperature=0.8
        )
        return json.loads(response.choices[0].message.content.strip())
    except Exception as e:
        print(f"大纲生成失败: {e}")
        return None

def generate_scene(world_setting, characters, scene_instruction, previous_summary):
    """阶段二：生成纯净的对话数组（确保每幕40轮以上）"""
    prompt = f"""你是一个长对话生成大师。请根据以下背景，撰写本幕戏的对话。
【世界观】：{world_setting}
【人物设定】：{characters}
【前情提要】：{previous_summary}
【本场剧情要求】：{scene_instruction}

强制性要求：
1. 必须包含至少 40 个对话节点。角色间的对话要有来回拉扯，不要进展太快。
2. 对话包含详细的动作或神态描写（写在内容里或括号里）。
3. 严格输出纯 JSON 数组，不要任何Markdown标记。格式必须严格如下：[
  {{"role": "旁白", "content": "场景描写..."}},
  {{"role": "角色A", "content": "（动作）台词..."}}
]"""
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        content = response.choices[0].message.content.strip()
        if content.startswith("```json"): content = content[7:-3].strip()
        elif content.startswith("```"): content = content[3:-3].strip()
        
        dialogues = json.loads(content)
        return dialogues
    except Exception as e:
        print(f"[错误] 对话生成失败: {e}")
        return None

def save_data(theme, outline, full_dialogues):
    """分离保存：主文件只存长对话，备份文件存世界观设定"""
    dialogues_data =[]
    if os.path.exists(DIALOGUES_FILE):
        with open(DIALOGUES_FILE, "r", encoding="utf-8") as f:
            try: dialogues_data = json.load(f)
            except: pass
            
    dialogue_record = {
        "theme": theme,
        "total_turns": len(full_dialogues),
        "dialogue": full_dialogues
    }
    dialogues_data.append(dialogue_record)
    with open(DIALOGUES_FILE, "w", encoding="utf-8") as f:
        json.dump(dialogues_data, f, ensure_ascii=False, indent=2)

    metadata =[]
    if os.path.exists(METADATA_FILE):
        with open(METADATA_FILE, "r", encoding="utf-8") as f:
            try: metadata = json.load(f)
            except: pass
            
    outline["theme"] = theme
    metadata.append(outline)
    with open(METADATA_FILE, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

def main():
    print(f"开始批量生成 100 个剧本！对话主文件将存入：{DIALOGUES_FILE}")

    completed_themes = set()
    if os.path.exists(DIALOGUES_FILE):
        with open(DIALOGUES_FILE, "r", encoding="utf-8") as f:
            try:
                existing_data = json.load(f)
                completed_themes = {item["theme"] for item in existing_data}
                print(f"检测到已完成 {len(completed_themes)} 个长对话，继续生成剩余的...")
            except:
                pass

    for index, theme in enumerate(THEMES):
        if theme in completed_themes:
            continue
            
        print(f"\n==========================================")
        print(f"任务 {index + 1}/{len(THEMES)}：{theme}")
        
        outline = generate_outline(theme)
        if not outline or "scenes_outline" not in outline:
            print("世界观大纲获取失败，跳过重试...")
            time.sleep(2)
            continue
            
        full_dialogues =[]
        current_summary = "（故事刚开始）"
        success_flag = True
        
        for scene_idx, scene_desc in enumerate(outline["scenes_outline"]):
            print(f"  -> 正在生成 第 {scene_idx + 1} 幕 对话...")
            
            scene_dialogues = generate_scene(
                outline.get("world_setting", ""), 
                outline.get("characters", ""), 
                scene_desc, 
                current_summary
            )
            
            if not scene_dialogues:
                success_flag = False
                break
                
            print(f"     完成！本幕生成了 {len(scene_dialogues)} 轮对话。")
            full_dialogues.extend(scene_dialogues)
            current_summary += f"\n【第{scene_idx + 1}幕剧情】：{scene_desc}"
            time.sleep(1) 
            
        if success_flag:
            save_data(theme, outline, full_dialogues)
            print(f"✅ 【{theme}】长对话(共 {len(full_dialogues)} 轮) 已成功保存！进度：{index + 1}/100")
        else:
            print(f"❌ 【{theme}】生成中断，放弃并进入下一个。")

if __name__ == "__main__":
    main()

开始批量生成 100 个剧本！对话主文件将存入：dialogues_dataset_100.json
检测到已完成 100 个长对话，继续生成剩余的...


二、增量提纲标注

In [ ]:
import json
import os
import time
from openai import OpenAI
API_KEY = "sk-732017ac4c85439fb123c6a954f124cd"
client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com/v1")

INPUT_FILE = "dialogues_dataset_100.json"
OUTPUT_FILE = "sft_training_data.json" 

CHUNK_SIZE = 30 
LIMIT_SCRIPTS = 100 

def generate_incremental_outline(history_outline, current_dialogue_text, attempt=1):
    """
    让教师模型根据历史提纲和最新对话，生成结构化状态 JSON。
    加入“记忆衰减”与“压缩合并”机制，防止事件列表无限膨胀。
    """
    system_prompt = "你是一个专业的长文本记忆状态管理器，专门为大语言模型提取关键增量记忆。"
    
    prompt = f"""请根据【历史提纲】和【最新对话】，提取并更新为【新的提纲】。
要求：
1. 精准识别对话中的状态转移、剧情推进、以及新增的关键物品或事件。
2. 【核心规则（记忆压缩）】：为了防止提纲过长，请对【历史提纲】中年代久远、不太重要的细枝末节进行高度概括或直接剔除。重点保留“对当前剧情仍有影响的核心伏笔”和“最新的关键进展”。

【历史提纲】：
{history_outline}

【最新对话】：
{current_dialogue_text}

必须严格输出合法的 JSON 格式，包含以下字段：
{{
  "summary": "本段最新对话的核心剧情一句话摘要",
  "character_states": {{
    "角色A": "当前的状态/情绪/关系变化（若无变化则不写或写无）",
    "角色B": "..."
  }},
  "key_items_or_events":[
    "线索1", 
    "事件2"
  ] // ⚠️注意：总事件条目数请严格控制在 20 条以内！旧事件请合并，新事件请精简。
}}
请直接输出纯 JSON 对象，不要包含 ```json 等任何 Markdown 标记。"""

    for i in range(3):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat", 
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content.strip()
            
            if content.startswith("```json"): content = content[7:-3].strip()
            elif content.startswith("```"): content = content[3:-3].strip()
            
            parsed_json = json.loads(content) 
            return json.dumps(parsed_json, ensure_ascii=False, indent=2)
            
        except json.JSONDecodeError:
            print(f"[警告] 格式错误，正在重试 ({i+1}/3)...")
            time.sleep(2)
        except Exception as e:
            print(f"      [错误] API 调用异常: {e}，正在重试...")
            time.sleep(2)
            
    return None

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"找不到输入文件 {INPUT_FILE}！请确认是否在同一目录下。")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        all_scripts = json.load(f)
    
    print(f"成功读取 {len(all_scripts)} 个剧本。本次计划处理前 {LIMIT_SCRIPTS} 个剧本。")

    sft_dataset =[]
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            try: 
                sft_dataset = json.load(f)
            except: 
                pass
            
    processed_themes = {item.get("_source_theme") for item in sft_dataset if "_source_theme" in item}

    for idx, script in enumerate(all_scripts[:LIMIT_SCRIPTS]):
     
        theme = script.get("theme", f"未知主题_{idx}")
        if theme in processed_themes:
            continue
            
        print(f"\n==========================================")
        print(f"正在标注剧本 {idx + 1}/{LIMIT_SCRIPTS}：{theme}")
        
        dialogues = script.get("dialogue",[])
        total_turns = len(dialogues)
        

        history_outline = "无（这是对话的开头）"
        

        for start_idx in range(0, total_turns, CHUNK_SIZE):
            end_idx = min(start_idx + CHUNK_SIZE, total_turns)
            chunk = dialogues[start_idx:end_idx]
            
    
            valid_lines =[]
            for d in chunk:
                if isinstance(d, dict):
                    role = d.get("role", d.get("name", "未知角色"))
                    content = d.get("content", d.get("text", d.get("dialogue", "")))
                    if content:  
                        valid_lines.append(f"{role}: {content}")
            
            chunk_text = "\n".join(valid_lines)
        
            if not chunk_text.strip():
                print("[警告] 遇到全是脏数据的切片，跳过本段。")
                continue
            
            print(f"  -> 处理切片: 第 {start_idx+1} 到 {end_idx} 轮...")
            
            new_outline_json_str = generate_incremental_outline(history_outline, chunk_text)
            
            if not new_outline_json_str:
                print("     [致命错误] 连续 3 次提取失败，跳过该剧本的后续切片。")
                break
                
            sft_record = {
                "instruction": "你是一个长对话记忆辅助模型。请根据【历史提纲】和【最新对话】，提取并更新为精简的状态提纲，重点保留对当前剧情有影响的核心伏笔和最新进展（输出JSON格式）。",
                "input": f"【历史提纲】:\n{history_outline}\n\n【最新对话】:\n{chunk_text}",
                "output": new_outline_json_str,
                "_source_theme": theme
            }
            
            sft_dataset.append(sft_record)
            
            history_outline = new_outline_json_str
            
            time.sleep(0.5) 
            
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(sft_dataset, f, ensure_ascii=False, indent=2)
            
        print(f"✅ 剧本【{theme}】标注完成，数据已安全追加到 {OUTPUT_FILE}。")

if __name__ == "__main__":
    main()

三、自动化质检与清洗

In [ ]:
import json
import os

INPUT_FILE = "sft_training_data.json"
OUTPUT_FILE = "final_sft_dataset.json" 

def main():
    print(f"开始执行提纲标注质量校验...")
    
    if not os.path.exists(INPUT_FILE):
        print(f"找不到文件 {INPUT_FILE}！")
        return

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        dataset = json.load(f)
        
    total_records = len(dataset)
    print(f"总计读取到 {total_records} 条 SFT 指令数据。\n")

    valid_data =[]
    error_count = {
        "json_parse_error": 0,
        "missing_keys_error": 0,
    }
    
    total_events_count = 0
    max_events_count = 0
    REQUIRED_KEYS = {"summary", "character_states", "key_items_or_events"}

    for idx, record in enumerate(dataset):
        output_str = record.get("output", "")
        
        try:
            parsed_output = json.loads(output_str)
        except json.JSONDecodeError:
            error_count["json_parse_error"] += 1
            continue 
            
        if not REQUIRED_KEYS.issubset(parsed_json_keys := set(parsed_output.keys())):
            error_count["missing_keys_error"] += 1
            continue  
            

        events = parsed_output.get("key_items_or_events",[])
        event_length = len(events) if isinstance(events, list) else 0
        total_events_count += event_length
        max_events_count = max(max_events_count, event_length)
        valid_data.append(record)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(valid_data, f, ensure_ascii=False, indent=2)

    print("="*40)
    print("🎯 数据集提纲质量校验报告")
    print("="*40)
    print(f"原始数据总量: {total_records} 条")
    print(f"✅ 校验通过 (高质量数据): {len(valid_data)} 条")
    print(f"❌ 剔除脏数据: {total_records - len(valid_data)} 条")
    print(f"   - JSON解析失败: {error_count['json_parse_error']} 条")
    print(f"   - 缺少关键字段: {error_count['missing_keys_error']} 条")
    print("-" * 40)
    if len(valid_data) > 0:
        print("🧠 记忆压缩效能分析:")
        print(f"   - 平均每条状态包含核心事件: {total_events_count / len(valid_data):.1f} 条")
        print(f"   - 单条状态最大事件数: {max_events_count} 条 (证明记忆衰减控制生效)")
    print("="*40)
    print(f"🎉 清洗完毕！")

if __name__ == "__main__":
    main()

四、测试集构建

In [ ]:
import json
import os
import time
from openai import OpenAI

API_KEY = "sk-732017ac4c85439fb123c6a954f124cd"
client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com/v1")
TEST_SCENARIOS = [
    {
        "id": 1,
        "theme": "科幻赛博：废弃金库的密码",
        "world_setting": "赛博朋克世界，两名雇佣兵（猎鹰、幻影）在废弃下城区执行潜入任务。",
        "scenes": [
            "第一幕：酒吧接头与准备。猎鹰送给幻影一块老旧的机械怀表作为护身符。特别强调怀表坏了，指针永远停在了【11:59】。两人闲聊一番后出发。",
            "第二幕：潜入大楼外围。两人小心翼翼地黑入安保系统，避开巡逻机甲。（注意：本幕绝不可提及怀表，专心描写潜入过程）。",
            "第三幕：遭遇财阀武装。潜入暴露，双方在走廊爆发极其惨烈的激光枪战，队友受伤。（注意：本幕绝不可提及怀表，专心描写战斗）。",
            "第四幕：下水道逃亡。两人逃入错综复杂的地下排污管道，躲避无人机的追杀，互相处理伤口。（注意：本幕绝不可提及怀表）。",
            "第五幕：金库绝境与密码。两人终于逃至地下金库大门前，后有追兵，门需要输入4位数字密码。关键时刻，幻影突然想起第一幕那块停在【11:59】的怀表，输入 1159，大门开启得救！"
        ]
    },
    {
        "id": 2,
        "theme": "仙侠修真：地摊上的废铁环",
        "world_setting": "修真界，主角（陆凡）与师妹（灵儿）闯荡江湖。",
        "scenes": [
            "第一幕：坊市淘宝。陆凡在天南坊市地摊上花【三块下品灵石】买了一个毫不起眼的【生锈黑铁指环】，摊主说这是从雷劫坑捡来的。",
            "第二幕：秘境开启。两人进入秘境，斩杀了一些低阶妖兽，采摘灵草。（注意：本幕绝不可提及铁环，专心写打怪探险）。",
            "第三幕：同门背叛。为了争夺一株千年灵芝，带队的师兄突然翻脸暗算他们。（注意：本幕绝不可提及铁环，专心写背叛与震惊）。",
            "第四幕：血林逃亡。陆凡带着受伤的师妹在血色森林中疯狂逃命，灵力即将耗尽。（注意：本幕绝不可提及铁环）。",
            "第五幕：玄雷绝境与神器觉醒。两人被魔修逼至悬崖，魔修发动致命的“九天玄雷大阵”。千钧一发之际，陆凡想起了第一幕买的【黑铁指环】，拿出来抵挡，指环吸收雷击觉醒为上古神器，反杀敌人！"
        ]
    },
    {
        "id": 3,
        "theme": "古代宫斗：玫瑰花粉过敏",
        "world_setting": "大楚后宫，女主（云妃）与贴身宫女（翠竹）步步为营，应对华贵妃的暗算。",
        "scenes": [
            "第一幕：御花园赏花。云妃明确向翠竹提到自己【对玫瑰花粉严重过敏】，一旦接触或食用就会起严重红疹甚至窒息。",
            "第二幕：中秋宫宴。众嫔妃在宴席上唇枪舌剑，华贵妃风头无两，云妃谨慎应对。（注意：本幕绝不可提及过敏和玫瑰，专心写宴会心机）。",
            "第三幕：内鬼疑云。翠竹发现云妃寝宫里有被人翻动的痕迹，两人暗中排查眼线。（注意：本幕绝不可提及过敏和玫瑰）。",
            "第四幕：雨夜冲突。华贵妃的太监在路上故意冲撞云妃的轿撵，双方发生激烈口角。（注意：本幕绝不可提及过敏和玫瑰）。",
            "第五幕：毒糕点反杀。华贵妃派人赏赐一盒极其精美的“西域凝香糕”以示和解。云妃察觉到糕点里暗藏了玫瑰花汁，回想起第一幕的过敏弱点，识破毒计，当场利用此事反将华贵妃一军！"
        ]
    },
    {
        "id": 4,
        "theme": "悬疑推理：诡异的童谣",
        "world_setting": "民国上海滩，侦探（沈探长）与助手（阿飞）调查富豪密室杀人案。",
        "scenes": [
            "第一幕：案发现场走访。两人走访外围时，无意中听到邻家小孩在唱一首奇怪的童谣：【“红色的鸟儿往南飞，敲了三下门”】。",
            "第二幕：审问嫌疑人。两人盘问死者的遗孀和生意合伙人，发现他们都在撒谎。（注意：本幕绝不可提及童谣，专心写推理盘问）。",
            "第三幕：黑市追踪。线索指向黑市，两人在赌场与地头蛇发生肢体冲突，险象环生。（注意：本幕绝不可提及童谣）。",
            "第四幕：秘密账本。他们在码头仓库找到了死者走私的账本，案情逐渐明朗。（注意：本幕绝不可提及童谣）。",
            "第五幕：解开机关箱。两人回到书房，找到了死者的机关密码箱。箱子没有锁，只有一个动物和方向转盘及按压按钮。沈探长猛然回想起第一幕的童谣，将转盘调到【红鸟、南向】并【敲击三次】，箱子打开！"
        ]
    },
    {
        "id": 5,
        "theme": "末日废土：收音机的频率",
        "world_setting": "丧尸爆发后的废土，小队队长（老狗）和新兵（石头）在城市废墟中寻找安全区。",
        "scenes": [
            "第一幕：公路搜刮。石头在废车里捡到一个破旧收音机，频道固定卡在了【FM 104.5】，老狗嘲笑他捡了个没用的废品。",
            "第二幕：废墟遭遇战。两人遇到小股丧尸，利用冷兵器艰难清理，收集弹药。（注意：本幕绝不可提及收音机，专心写末日生存）。",
            "第三幕：掠夺者伏击。在超市寻找食物时，遭遇了另一群暴徒的伏击，双方展开激烈枪战。（注意：本幕绝不可提及收音机）。",
            "第四幕：雨夜狂奔。枪声引来了庞大的尸潮，两人只能放弃物资，在大雨中向高处逃亡。（注意：本幕绝不可提及收音机）。",
            "第五幕：广播塔求生。两人被海量丧尸围困在广播信号塔内。老狗发现可以通过基站发射高频声波引开丧尸，但不知道安全频率。石头瞬间回想起第一幕收音机卡死的频率，调至【104.5】，成功引走尸潮生还！"
        ]
    }
]

def generate_scene(theme, world_setting, scene_outline, previous_summary):
    prompt = f"""你是一个专业剧本作家。请根据以下大纲，撰写【这一幕】的长对话。
【世界观】：{theme} - {world_setting}
【前情提要】：{previous_summary}
【本幕剧情要求】：{scene_outline}

强制性要求：
1. 这一幕的对话必须非常长！必须包含至少 50 到 60 个对话节点（角色拉扯、探索、心理活动等）。
2. 严格遵循剧情要求，如果要求中说“绝不可提及某物”，则绝对不能让角色提到它。
3. 严格输出合法的 JSON 数组，格式如下：[
  {{"role": "旁白", "content": "场景描写..."}},
  {{"role": "角色A", "content": "（动作）台词..."}}
]
请直接输出纯 JSON 数组，不要任何 Markdown 标记。"""

    for _ in range(3):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7
            )
            content = response.choices[0].message.content.strip()
            # 清理 Markdown 代码块
            if content.startswith("```json"): content = content[7:-3].strip()
            elif content.startswith("```"): content = content[3:-3].strip()
            
            dialogues = json.loads(content)
            return dialogues
        except Exception as e:
            print(f"      [警告] 生成异常 ({e})，正在重试...")
            time.sleep(2)
            
    return None

def main():
    print("开始生成超长伏笔测试集（每剧本约300轮）...\n")
    
    for scenario in TEST_SCENARIOS:
        file_name = f"test_dataset({scenario['id']}).json"

        if os.path.exists(file_name):
            print(f"文件 {file_name} 已存在，跳过。")
            continue
            
        print(f"==========================================")
        print(f"正在生成剧本 {scenario['id']}/5：【{scenario['theme']}】")
        
        full_dialogues = []
        current_summary = "（故事刚开始）"
        success_flag = True
        
        for scene_idx, scene_outline in enumerate(scenario["scenes"]):
            print(f"  -> 正在生成 第 {scene_idx + 1} 幕 ...")
            
            scene_dialogues = generate_scene(
                scenario["theme"], 
                scenario["world_setting"], 
                scene_outline, 
                current_summary
            )
            
            if not scene_dialogues:
                print(f"     ❌ 第 {scene_idx + 1} 幕生成彻底失败。")
                success_flag = False
                break
                
            print(f"     ✅ 完成！本幕生成了 {len(scene_dialogues)} 轮对话。")
            full_dialogues.extend(scene_dialogues)

            current_summary += f"\n【第{scene_idx + 1}幕发生的事】：{scene_outline}"
            time.sleep(1)
            
        if success_flag:
            output_data = [{
                "theme": scenario["theme"],
                "total_turns": len(full_dialogues),
                "dialogue": full_dialogues
            }]
            
            with open(file_name, "w", encoding="utf-8") as f:
                json.dump(output_data, f, ensure_ascii=False, indent=2)
            print(f"🎉 剧本 {scenario['id']} 生成完毕！总轮数: {len(full_dialogues)} 轮。已保存为 {file_name}\n")
        else:
            print(f"❌ 剧本 {scenario['id']} 生成中断。\n")

if __name__ == "__main__":
    main()